# UAV Battery Tool — Notebook 05: Report Generation

Generate formatted Excel reports from simulation results, battery comparisons, and flight log analysis.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

from batteries.database import BatteryDatabase
from batteries.log_importer import generate_synthetic_log
from batteries.parameter_fitter import fit_all, apply_fitted_params
from mission.simulator import run_simulation, compare_batteries, temperature_sweep
from mission.report_generator import generate_report

DB_PATH = '../battery_db.xlsx'
db = BatteryDatabase(DB_PATH).load()
print(db.summary())

## 1 · Configure Report Parameters

In [ ]:
# ── Select packs, mission, and UAV ────────────────────────────────────────────
PRIMARY_PACK_ID    = 'BAT_MID_6S2P'
COMPARE_PACK_IDS   = ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P']
MISSION_ID         = 'SURVEY_STD'
UAV_ID             = 'HEX_SURVEY_900'
AMBIENT_TEMP_C     = 25.0
TEMP_SWEEP         = [-20,-15,-10,-5,0,5,10,15,20,25,30,35,40,45]
INCLUDE_LOG        = True   # generate synthetic log and fit parameters
OUT_PATH           = 'UAV_Battery_Report.xlsx'
# ─────────────────────────────────────────────────────────────────────────────

primary_pack = db.packs[PRIMARY_PACK_ID]
mission      = db.missions[MISSION_ID]
uav          = db.uav_configs[UAV_ID]
compare_packs= [db.packs[pid] for pid in COMPARE_PACK_IDS if pid in db.packs]
print(f'Primary pack  : {primary_pack}')
print(f'Compare packs : {[p.battery_id for p in compare_packs]}')
print(f'Mission       : {mission}')

## 2 · Run Simulations

In [ ]:
print('Running simulations...')

# Primary simulation
primary_result = run_simulation(
    pack=primary_pack, mission=mission, uav=uav,
    discharge_pts=db.discharge_pts,
    ambient_temp_c=AMBIENT_TEMP_C, dt_s=1.0
)
print(primary_result.summary())

# Multi-pack comparison
compare_results = compare_batteries(
    packs=compare_packs, mission=mission, uav=uav,
    discharge_pts=db.discharge_pts,
    ambient_temp_c=AMBIENT_TEMP_C, dt_s=2.0
)
print(f'\nMulti-pack results:')
for r in compare_results:
    print(f'  {r.pack_id}: SoC={r.final_soc:.1f}%  Vmin={r.min_voltage:.3f}V  depleted={r.depleted}')

In [ ]:
# Temperature sweep
print(f'Running temperature sweep ({len(TEMP_SWEEP)} temperatures)...')
sweep_results = temperature_sweep(
    pack=primary_pack, mission=mission, uav=uav,
    discharge_pts=db.discharge_pts,
    temperatures_c=TEMP_SWEEP, dt_s=5.0
)
print('Done.')

df_sweep = pd.DataFrame([{
    'Temp (C)': t, 'Final SoC (%)': round(r.final_soc,1),
    'Peak sag (V)': round(r.peak_sag_v,3), 'Min V (V)': round(r.min_voltage,3),
    'Max T (C)': round(r.max_temp_c,1), 'Depleted': r.depleted}
    for t, r in zip(TEMP_SWEEP, sweep_results)])
print(df_sweep.to_string(index=False))

## 3 · Flight Log Analysis (optional)

In [ ]:
log = fitted = None
if INCLUDE_LOG:
    from batteries.log_importer import generate_synthetic_log
    from batteries.parameter_fitter import fit_all, apply_fitted_params
    print('Generating synthetic flight log and fitting parameters...')
    log = generate_synthetic_log(primary_pack, mission, uav, db.discharge_pts,
                                  ambient_temp_c=AMBIENT_TEMP_C, dt_s=2.0,
                                  noise_v=0.03, noise_i=0.8)
    fitted = fit_all(log, primary_pack.pack_capacity_ah,
                     primary_pack.pack_voltage_cutoff,
                     chem_id=primary_pack.chemistry_id,
                     pack_id=primary_pack.battery_id)
    print(fitted.summary())
else:
    print('Skipping log analysis (INCLUDE_LOG=False)')

## 4 · Generate Excel Report

In [ ]:
print(f'Generating report: {OUT_PATH}')
report_path = generate_report(
    out_path=OUT_PATH,
    results=compare_results,
    packs=compare_packs,
    mission=mission,
    uav_name=UAV_ID,
    ambient_temp_c=AMBIENT_TEMP_C,
    temp_sweep_temps=TEMP_SWEEP,
    temp_sweep_results=sweep_results,
    flight_log=log,
    fitted_params=fitted,
    primary_pack=primary_pack,
)
print(f'Report saved: {report_path}')

## 5 · Quick Battery Selection Table

Which batteries from the catalog can safely complete this mission at all temperature sweep points?

In [ ]:
from mission.simulator import SimulationResult
all_packs = list(db.packs.values())
selection_rows = []
print(f'Testing {len(all_packs)} packs across {len(TEMP_SWEEP)} temperatures...')
for p in all_packs:
    row = {'Pack_ID': p.battery_id, 'Chemistry': p.chemistry_id,
           'Energy (Wh)': p.pack_energy_wh, 'Weight (g)': p.pack_weight_g,
           'Wh/kg': p.specific_energy_wh_kg}
    pass_all = True
    for temp in [-10, 0, 15, 25, 35]:
        try:
            r = run_simulation(p, mission, uav, db.discharge_pts,
                               ambient_temp_c=temp, dt_s=10.0)
            row[f'{temp}C SoC'] = round(r.final_soc, 1)
            row[f'{temp}C pass'] = not r.depleted and r.final_soc > 15
            if r.depleted or r.final_soc <= 15: pass_all = False
        except Exception as e:
            row[f'{temp}C SoC'] = 'ERR'; row[f'{temp}C pass'] = False
            pass_all = False
    row['All temps PASS'] = pass_all
    selection_rows.append(row)

df_sel = pd.DataFrame(selection_rows).set_index('Pack_ID')
df_sel.sort_values('All temps PASS', ascending=False)